In [1]:
from ml.data import load_split
X_train, X_test, y_train, y_test = load_split()
print(len(X_train), len(X_test))

24000 6000


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from ml.preprocessing import clean_text, STOP_WORDS

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=clean_text, stop_words=STOP_WORDS,
                              max_features=50000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 5],
    "clf__C": [0.5, 1, 2, 5],
}

search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=3)
search.fit(X_train, y_train)

print("Best settings:", search.best_params_)
print("Best CV Macro-F1:", round(search.best_score_, 3))

Best settings: {'clf__C': 1, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Best CV Macro-F1: 0.632


In [3]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score

def make(penalty):
    clf = LogisticRegression(penalty=penalty, solver="saga", C=1,
                             max_iter=2000, class_weight="balanced")
    return Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=clean_text, stop_words=STOP_WORDS,
                                  ngram_range=(1, 2), min_df=5, max_features=50000)),
        ("clf", clf)])

for penalty in ["l2", "l1"]:
    model = make(penalty)
    pred = cross_val_predict(model, X_train, y_train, cv=3)
    model.fit(X_train, y_train)
    coef = model.named_steps["clf"].coef_
    print(penalty.upper(),
          "| Macro-F1:", round(f1_score(y_train, pred, average="macro"), 3),
          "| zero weights:", int((coef == 0).sum()), "of", coef.size)

c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of 

L2 | Macro-F1: 0.512 | zero weights: 0 of 86802


c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\LOQ\OneDrive\Documents\Project\venv\Lib\site-packages\sklearn\linear_m

KeyboardInterrupt: 